# 04 — Content Pipeline (Exploration First)
Explore → Profile → Correlate → Validate References

This notebook performs **deep analysis only** for `object_type == "content"` before we design extractors/loaders.

Focus areas requested:
- `content_type` and `content_sub_type` (and their correlation)
- Core fields for future `:Content` node
- `content_site_id` mapping to existing `:Site`
- `content_files` mapping to non-image `:File`
- `file_content_id` reconciliation from files side
- `SitePageCategory` as inferred entity scoped by site
- Keep `_aggregated_locations` as-is for now
- **Do not ingest/analyze deeply** `content_topics` now (defer)


## 0 · Setup

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD

print(f"Project root      : {PROJECT_ROOT}")
print(f"Raw file          : {RAW_FILE}")
print(f"Raw file exists   : {RAW_FILE.exists()}")
print(f"Normalized dir    : {NORMALIZED_DIR}")
print(f"Normalized exists : {NORMALIZED_DIR.exists()}")

Project root      : /home/ubuntu/graph_experiments
Raw file          : /home/ubuntu/graph_experiments/data/raw/backyard_dump_19022026_172648.jsonl
Raw file exists   : True
Normalized dir    : /home/ubuntu/graph_experiments/data/normalized
Normalized exists : True


## 1 · Explore Raw Data

In [2]:
# Load all relevant object types for cross-reference checks
content_items = []
people_ids = set()
site_ids = set()
raw_non_image_file_ids = set()

with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        ot = obj.get("object_type")

        if ot == "content":
            content_items.append(obj)
        elif ot == "people" and obj.get("id"):
            people_ids.add(obj["id"])
        elif ot == "site" and obj.get("id"):
            site_ids.add(obj["id"])
        elif ot == "file" and obj.get("id") and obj.get("file_is_image") is not True:
            raw_non_image_file_ids.add(obj["id"])

print(f"Content records           : {len(content_items)}")
print(f"People IDs (raw)          : {len(people_ids)}")
print(f"Site IDs (raw)            : {len(site_ids)}")
print(f"Non-image File IDs (raw)  : {len(raw_non_image_file_ids)}")

Content records           : 3142
People IDs (raw)          : 954
Site IDs (raw)            : 45
Non-image File IDs (raw)  : 555


In [3]:
# Save 10 sample content records for debugging
sample_path = PROJECT_ROOT / "data" / "debug" / "sample_contents.jsonl"
sample_path.parent.mkdir(parents=True, exist_ok=True)
with open(sample_path, "w") as out:
    for item in content_items[:10]:
        out.write(json.dumps(item) + "\n")
print(f"Saved {min(10, len(content_items))} samples → {sample_path}")

# Pretty-print one sample (truncate heavy fields)
sample = dict(content_items[0])
for heavy_key in ["content_topics", "_aggregated_locations", "content_files"]:
    if heavy_key in sample and isinstance(sample[heavy_key], list) and len(sample[heavy_key]) > 2:
        sample[heavy_key] = sample[heavy_key][:2] + ["...[truncated]..."]
print("\n--- Sample content record ---")
print(json.dumps(sample, indent=2))

Saved 10 samples → /home/ubuntu/graph_experiments/data/debug/sample_contents.jsonl

--- Sample content record ---
{
  "id": "03e5a566-4785-4e48-8b07-23698ff5cf11",
  "autocomplete": "Add-On Products: Recognition Best Practice",
  "content_title": "Add-On Products: Recognition Best Practice",
  "content_type": "page",
  "content_site_id": "906512a7-0e25-40f5-9ace-375b5a7b59e5",
  "content_site_name": "Customer Success Managers",
  "content_site_img_url": "/content/o/b8242631-a6c6-4867-a714-ebdd8b46abc3",
  "content_site_type": "public",
  "content_site_has_pages": true,
  "content_site_has_albums": true,
  "content_site_has_events": true,
  "content_site_is_active": true,
  "content_site_is_deleted": false,
  "content_body": "{\"type\":\"doc\",\"content\":[{\"type\":\"heading\",\"attrs\":{\"indentation\":0,\"textAlign\":\"left\",\"level\":3,\"className\":\"\",\"id\":null},\"content\":[{\"type\":\"text\",\"marks\":[{\"type\":\"bold\"}],\"text\":\"Overview\"}]},{\"type\":\"paragraph\",\"a

In [4]:
# Key presence and null analysis
all_keys = set()
for item in content_items:
    all_keys.update(item.keys())

n = len(content_items)
print(f"Total distinct keys in content records: {len(all_keys)}")
print()
print(f"{'Key':<40} {'Present':>10} {'Null':>10} {'Non-empty':>10}")
print("-" * 76)
for k in sorted(all_keys):
    present = sum(1 for s in content_items if k in s)
    nulls = sum(1 for s in content_items if k in s and s.get(k) is None)
    non_empty = sum(1 for s in content_items if s.get(k) not in (None, '', []))
    print(f"{k:<40} {present:>10} {nulls:>10} {non_empty:>10}")

Total distinct keys in content records: 67

Key                                         Present       Null  Non-empty
----------------------------------------------------------------------------
_aggregated_locations                          3107          0       2209
autocomplete                                   3142          0       3142
content_all_day_event                           298          0        298
content_audiences                              3142          0         17
content_authored_by_name                       3142          0       3142
content_body                                   3142          0       3142
content_dismiss_validation_warning             3142       3019        123
content_event_ends_at                           298          0        298
content_event_starts_at                         298          0        298
content_event_timezone_id                      3107       2815        292
content_event_timezone_name                    3113       2815   

### 1.1 `content_type` / `content_sub_type` distribution and correlation

In [12]:
print("=== content_type ===")
for k, v in Counter(s.get("content_type") for s in content_items).most_common():
    print(f"  {repr(k)}: {v}")

print("\n=== content_sub_type ===")
for k, v in Counter(s.get("content_sub_type") for s in content_items).most_common():
    print(f"  {repr(k)}: {v}")

pair_counter = Counter((s.get("content_type"), s.get("content_sub_type")) for s in content_items)
print("\n=== Top type/sub_type pairs ===")
for (ct, st), v in pair_counter.most_common(20):
    print(f"  {ct!r} | {st!r}: {v}")

subtype_to_types = defaultdict(set)
for s in content_items:
    subtype_to_types[s.get("content_sub_type")].add(s.get("content_type"))
conflicts = {sub: types for sub, types in subtype_to_types.items() if sub is not None and len(types) > 1}

print(f"\nSub-types mapping to multiple content_type values: {len(conflicts)}")
if conflicts:
    for sub, types in sorted(conflicts.items(), key=lambda x: len(x[1]), reverse=True):
        print(f"  {sub!r}: {sorted(types)}")

=== content_type ===
  'page': 2813
  'event': 298
  'album': 31

=== content_sub_type ===
  'knowledge': 1724
  'news': 1089
  None: 329

=== Top type/sub_type pairs ===
  'page' | 'knowledge': 1724
  'page' | 'news': 1089
  'event' | None: 298
  'album' | None: 31

Sub-types mapping to multiple content_type values: 0


### 1.2 Core field distributions (candidate `:Content` properties)

In [14]:
print("content_is_featured:", Counter(s.get("content_is_featured") for s in content_items))
print("content_is_must_read:", Counter(s.get("content_is_must_read") for s in content_items))

validated_non_null = sum(1 for s in content_items if s.get("content_validated_on") not in (None, ""))
print(f"content_validated_on non-null: {validated_non_null}/{len(content_items)}")

agg_non_null = sum(1 for s in content_items if s.get("_aggregated_locations") not in (None, [], ""))
print(f"_aggregated_locations non-empty: {agg_non_null}/{len(content_items)}")

print("\nNOTE: `content_topics` is explicitly deferred from extraction/ingestion for now.")
content_topics_non_empty = sum(1 for s in content_items if s.get("content_topics") not in (None, [], ""))
print(f"content_topics non-empty: {content_topics_non_empty}/{len(content_items)}")

content_is_featured: Counter({False: 3141, True: 1})
content_is_must_read: Counter({False: 3098, None: 35, True: 9})
content_validated_on non-null: 3107/3142
_aggregated_locations non-empty: 2209/3142

NOTE: `content_topics` is explicitly deferred from extraction/ingestion for now.
content_topics non-empty: 1486/3142


### 1.3 Cross-entity reference checks (`:Site`, `:People`)

In [7]:
site_ref_ids = set(s["content_site_id"] for s in content_items if s.get("content_site_id"))
missing_site_refs = sorted(site_ref_ids - site_ids)
print(f"Unique content_site_id: {len(site_ref_ids)}")
print(f"Missing content_site_id in Site objects: {len(missing_site_refs)}")
if missing_site_refs:
    print("  sample missing:", missing_site_refs[:20])

for fk in ["createdbyid", "lastmodifiedbyid"]:
    refs = set(s[fk] for s in content_items if s.get(fk))
    missing = sorted(refs - people_ids)
    print(f"\n{fk} unique refs: {len(refs)}")
    print(f"{fk} missing in People objects: {len(missing)}")
    if missing:
        print("  sample missing:", missing[:20])

Unique content_site_id: 39
Missing content_site_id in Site objects: 0

createdbyid unique refs: 272
createdbyid missing in People objects: 0

lastmodifiedbyid unique refs: 146
lastmodifiedbyid missing in People objects: 0


### 1.4 Inferred entity analysis: `SitePageCategory` (site-scoped)

In [8]:
# Category pair checks
cat_records = [
    s for s in content_items
    if s.get("content_page_category_id") or s.get("content_page_category_name")
]

id_to_names = defaultdict(set)
name_to_ids = defaultdict(set)
site_scoped_keys = set()

for s in cat_records:
    cid = s.get("content_page_category_id")
    cname = s.get("content_page_category_name")
    sid = s.get("content_site_id")

    if cid is not None:
        id_to_names[cid].add(cname)
    if cname is not None:
        name_to_ids[cname].add(cid)

    # site-scoped identity candidate
    if sid and cid and cname:
        site_scoped_keys.add((sid, cid, cname))

id_conflicts = {cid: names for cid, names in id_to_names.items() if len(names) > 1}
name_conflicts = {name: ids for name, ids in name_to_ids.items() if len(ids) > 1}

missing_both = sum(1 for s in content_items if not s.get("content_page_category_id") and not s.get("content_page_category_name"))
only_id = sum(1 for s in content_items if s.get("content_page_category_id") and not s.get("content_page_category_name"))
only_name = sum(1 for s in content_items if s.get("content_page_category_name") and not s.get("content_page_category_id"))

print(f"Rows with category signal (id or name): {len(cat_records)}")
print(f"Distinct site-scoped (site,id,name) keys: {len(site_scoped_keys)}")
print(f"Category id→name conflicts: {len(id_conflicts)}")
print(f"Category name→id conflicts (global): {len(name_conflicts)}")
print(f"Missing both category id+name: {missing_both}")
print(f"Only id present: {only_id}")
print(f"Only name present: {only_name}")

if name_conflicts:
    print("\nSample name→id conflicts (suggests site scoping is important):")
    for i, (name, ids) in enumerate(list(name_conflicts.items())[:20], 1):
        print(f"  {i:>2}. {name!r} -> {sorted(ids)}")

Rows with category signal (id or name): 2813
Distinct site-scoped (site,id,name) keys: 279
Category id→name conflicts: 0
Category name→id conflicts (global): 20
Missing both category id+name: 329
Only id present: 0
Only name present: 0

Sample name→id conflicts (suggests site scoping is important):
   1. 'Product Best Practices' -> ['75aed90e-3387-4681-8539-51abd306e093', 'bb4152c4-f28d-44e7-a60b-7ee378883bba']
   2. 'Integrations' -> ['a529c6d0-5164-4e91-a4ff-226082c6eb12', 'ee8cd052-7555-40d2-8ec4-e98df91e1d9e']
   3. 'Uncategorized' -> ['02930a2b-c046-4fb7-a98f-fdf9b61cde08', '09134c6e-4347-438b-aeba-f3a9d437462a', '2ef865e8-73ec-40af-bd10-f2045a7030dd', '3868f891-ecac-470f-8eeb-bc93a2a13c4a', '3947fda1-c39e-4552-a524-3bdf6751e782', '3d514093-a349-4f1f-a0c1-02a6c6e6d4b0', '43de2b38-e9d5-412e-9ea8-81ccbb16f1f5', '4754215e-399f-4bc1-b2d0-b6ecb80612e1', '48b165f0-17e8-4675-855f-de48d10ed670', '4e0b9f67-8adc-42c1-a3bb-bd2d08befade', '53d31727-6a45-492b-aabb-5b73f2993b4e', '62cd6243-5835

### 1.5 `content_files` mapping and reconciliation with non-image files

In [9]:
# Load normalized file ids if available
normalized_file_ids = set()
norm_file_path = NORMALIZED_DIR / "files.jsonl"
if norm_file_path.exists():
    with open(norm_file_path) as f:
        for line in f:
            obj = json.loads(line)
            if obj.get("id"):
                normalized_file_ids.add(obj["id"])

print(f"Normalized non-image file IDs available: {len(normalized_file_ids)}")

# content_files structure inspection
shape_counter = Counter()
shape_examples = {}

for s in content_items:
    cf = s.get("content_files")
    if isinstance(cf, list):
        if not cf:
            key = "list_empty"
        else:
            key = f"list_of_{type(cf[0]).__name__}"
        shape_counter[key] += 1
        shape_examples.setdefault(key, cf[:2])
    elif cf is None:
        shape_counter["none"] += 1
    else:
        key = type(cf).__name__
        shape_counter[key] += 1
        shape_examples.setdefault(key, cf)

print("\ncontent_files shapes:")
for k, v in shape_counter.items():
    print(f"  {k}: {v}")

print("\nSample content_files payloads:")
for k, v in shape_examples.items():
    print(f"  {k}: {v}")

# Extract file refs from content_files
content_file_refs = []
for s in content_items:
    cf = s.get("content_files")
    if not isinstance(cf, list):
        continue
    for item in cf:
        if isinstance(item, str):
            if item:
                content_file_refs.append(item)
        elif isinstance(item, dict):
            fid = item.get("file_id") or item.get("id") or item.get("content_file_id")
            if fid:
                content_file_refs.append(fid)

content_file_ref_set = set(content_file_refs)

resolved_in_norm = sum(1 for fid in content_file_refs if fid in normalized_file_ids)
resolved_in_raw_non_image = sum(1 for fid in content_file_refs if fid in raw_non_image_file_ids)
missing_non_image = sorted(content_file_ref_set - raw_non_image_file_ids)

print("\n=== content_files reconciliation ===")
print(f"Total refs extracted                 : {len(content_file_refs)}")
print(f"Unique refs extracted                : {len(content_file_ref_set)}")
print(f"Refs found in normalized files       : {resolved_in_norm}")
print(f"Refs found in raw non-image files    : {resolved_in_raw_non_image}")
print(f"Unique refs missing in non-image set : {len(missing_non_image)}")
if missing_non_image:
    print(f"Missing sample: {missing_non_image[:25]}")

Normalized non-image file IDs available: 555

content_files shapes:
  list_of_dict: 1144
  list_empty: 1980
  none: 18

Sample content_files payloads:
  list_of_dict: [{'file_id': '26b4b697-62e1-4e20-b424-3fde3c5ea6ab', 'file_name': 'mceclip1.png', 'file_url': '/content/r/26b4b697-62e1-4e20-b424-3fde3c5ea6ab', 'file_thumbnail_url': None, 'file_mime_type': 'image/png', 'file_type': 'PNG'}]
  list_empty: []

=== content_files reconciliation ===
Total refs extracted                 : 2910
Unique refs extracted                : 2899
Refs found in normalized files       : 116
Refs found in raw non-image files    : 116
Unique refs missing in non-image set : 2788
Missing sample: ['00285705-17ec-4bd3-9584-b692eaf490e9', '003fc3cf-82a2-42b3-8fd9-0c702d614a8b', '009002e0-dcdd-4e88-a5d0-a1142d429158', '00a5b5e3-b832-4c76-b63d-60852c3a8452', '00a7f258-ec07-4dd2-9b5b-232352d453cb', '00c5bb37-884c-4738-80d7-040775e7666e', '00ded979-bc27-4aa3-9af8-558dc3ac2cae', '00e64d95-18c3-40a3-b61e-45abc2834874'

### 1.6 Reverse link check from file side (`file_content_id`)

In [15]:
content_ids = set(s.get("id") for s in content_items if s.get("id"))

non_image_file_content_ids = set()
with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("object_type") == "file" and obj.get("file_is_image") is not True:
            fcid = obj.get("file_content_id")
            if fcid:
                non_image_file_content_ids.add(fcid)

matched = non_image_file_content_ids & content_ids
missing_content = sorted(non_image_file_content_ids - content_ids)

print(f"Unique non-image file_content_id values: {len(non_image_file_content_ids)}")
print(f"file_content_id values matching content.id: {len(matched)}")
print(f"file_content_id values missing in content set: {len(missing_content)}")
if missing_content:
    print(f"Missing sample: {missing_content[:25]}")

print("\nInterpretation:")
print("- Keep file_content_id as a property for now (as decided).")
print("- Later content pipeline can reconcile both signals:")
print("  (a) content.content_files[*].file_id and (b) file.file_content_id")

Unique non-image file_content_id values: 233
file_content_id values matching content.id: 22
file_content_id values missing in content set: 211
Missing sample: ['100421', '102895', '103056', '103070', '103087', '103283', '104497', '104518', '105566', '107959', '108949', '108953', '108974', '108987', '109042', '109050', '119903', '119965', '124680', '13802', '13963', '14127', '14469', '14492', '14495']

Interpretation:
- Keep file_content_id as a property for now (as decided).
- Later content pipeline can reconcile both signals:
  (a) content.content_files[*].file_id and (b) file.file_content_id


### 1.7 DataFrame overview (core fields)

In [11]:
core_cols = [
    "id",
    "content_title",
    "content_type",
    "content_sub_type",
    "content_site_id",
    "content_is_featured",
    "content_validated_on",
    "content_is_must_read",
    "createdbyid",
    "lastmodifiedbyid",
    "createddate",
    "lastmodifieddate",
    "content_page_category_id",
    "content_page_category_name",
    "_aggregated_locations",
    "content_files",
    "content_topics",
]

content_df = pd.DataFrame([{k: s.get(k) for k in core_cols} for s in content_items])
print(f"Shape: {content_df.shape}")
content_df.head(10)

Shape: (3142, 17)


,id,content_title,content_type,content_sub_type,content_site_id,content_is_featured,content_validated_on,content_is_must_read,createdbyid,lastmodifiedbyid,createddate,lastmodifieddate,content_page_category_id,content_page_category_name,_aggregated_locations,content_files,content_topics
0,03e5a566-4785-4e48-8b07-23698ff5cf11,Add-On Products: Recognition Best Practice,page,knowledge,906512a7-0e25-40f5-9ace-375b5a7b59e5,False,2025-01-04T20:59:21.746Z,False,9a8faf20-5bcf-4fef-929f-fcd4e1283b4d,e8bcab12-2978-4c65-a511-8d6d62c27102,2024-12-19T18:12:00.285Z,2025-03-26T22:33:58.825Z,bb4152c4-f28d-44e7-a60b-7ee378883bba,Product Best Practices,[global],[{'file_id': '26b4b697-62e1-4e20-b424-3fde3c5e...,[]
1,03e890aa-0654-46f4-b0ec-464d0e715bf2,Best Practices: Segments (SFDC),page,knowledge,906512a7-0e25-40f5-9ace-375b5a7b59e5,False,2024-01-31T07:00:00.000Z,False,acc88c24-7434-47d7-bd00-69b0890b9af8,e9114dcb-819d-406a-b177-b0d7dfe805fd,2024-04-18T18:54:27.220Z,2025-03-26T22:50:02.307Z,bb4152c4-f28d-44e7-a60b-7ee378883bba,Product Best Practices,[global],[],[{'id': 'e17bbdd4-0b00-4c82-b016-1ef7a5464489'...
2,03eb64e8-1e46-41ab-9c18-995faf0ac087,Pro Tip: Win Reports in Backyard,page,knowledge,7db47563-7c6a-4206-bf10-3e736f7d2804,False,2025-07-22T14:34:33.276Z,False,df2bea29-6956-42bf-9c2f-67cf081f39da,df2bea29-6956-42bf-9c2f-67cf081f39da,2024-08-06T20:01:10.326Z,2025-12-05T02:11:11.508Z,378c108e-40dc-4470-a9c1-5013ee7576e0,Pro-Tips,[],[{'file_id': 'eb04a0af-2524-4cb3-bc58-7692757f...,[{'id': '61326e45-71ff-4775-9a82-fc019ff23885'...
3,03f48976-1a5f-4624-b8de-dd2d2c3795bf,"LumApps Webinar ""Measuring the Impact of Your ...",page,knowledge,39b3d853-fcc5-43cd-a508-d77f56bb0b02,False,2025-07-30T19:19:45.369Z,False,3a331b59-add7-4198-a9a8-78c11af00261,3a331b59-add7-4198-a9a8-78c11af00261,2024-08-26T20:01:45.645Z,2025-07-30T19:19:45.369Z,ddbc8032-132f-44fc-9ed6-4b4803fa50b1,Research,[global],[{'file_id': 'a9616982-c269-4d4f-90f5-eb253594...,[{'id': 'dd54a19b-b790-4dac-a3d1-376841f9bfff'...
4,04060b92-c2be-4f45-a802-6dd3c198f89a,Chatbot Integration- Aisera,page,knowledge,73746744-c930-4597-9c7d-9723c18c7d07,False,2025-02-18T16:11:00.470Z,False,86e941b2-44d3-4424-a637-809d0122d5a6,3f1dcf21-0489-46de-a743-1a47b506e795,2024-03-26T19:15:31.948Z,2025-11-19T12:45:50.362Z,ee8cd052-7555-40d2-8ec4-e98df91e1d9e,Integrations,[global],[],[]
5,043c5116-edf7-46f4-b923-fbcaf600d768,Customer Win: FOO,page,news,0d655175-49af-4d4d-a6f4-9701c94cd198,False,2024-10-29T15:53:00.000Z,False,7e1e21d4-6949-4f21-9a13-a73b346f7c22,7e1e21d4-6949-4f21-9a13-a73b346f7c22,2024-10-29T15:54:05.314Z,2024-11-12T02:00:57.196Z,8a1f530e-527e-439a-a404-989c36823179,Sales Wins,[mena],[],[]
6,046a296c-d72a-47e4-9ef4-2bf98b29e86b,Getting started on Usertesting.com,page,knowledge,b7282750-8c3d-4ae3-b935-39e69dc3f803,False,2025-07-15T13:02:00.000Z,False,fa4f7f63-5c35-40a8-ad51-f0f982706f27,89f6fc97-6447-4215-9918-4ec35eab1a9f,2025-07-15T13:58:32.836Z,2025-12-16T05:05:53.942Z,982d7983-5bc0-4c05-bb19-feb255289a8c,Uncategorized,[global],[{'file_id': 'b7590f92-b356-47f3-93cc-f0439bd6...,[{'id': '90cdf8e1-12d0-4d9b-994a-bfbf8fcfc1ba'...
7,0491a334-4ab0-45c1-b902-d01e8924a138,Unite Us Success Story,page,news,0d655175-49af-4d4d-a6f4-9701c94cd198,False,2022-12-11T18:30:00.000Z,False,7e1e21d4-6949-4f21-9a13-a73b346f7c22,7e1e21d4-6949-4f21-9a13-a73b346f7c22,2025-01-22T16:21:08.794Z,2025-02-05T02:01:33.412Z,5db2fc96-fb25-4d77-a9db-6d44153730fc,Success Stories,[united states],[],[]
8,04a254e0-da56-4c40-a9e9-fb70db5fdf0a,Connect Simpplr New York,event,NaN,36ca1fd3-3eaa-488f-b003-755394163cbb,False,2024-04-10T14:39:00.000Z,False,61816cfb-4e12-450f-90c3-795d4109d498,61816cfb-4e12-450f-90c3-795d4109d498,2024-04-10T14:40:58.946Z,2025-08-15T02:03:48.926Z,NaN,NaN,[united states],[],[]
9,04acc0c5-f3a7-4d55-8aec-72a1225c1131,Best practices guide: Using the My Team dashboard,page,knowledge,b40bbe83-c93c-41fe-a08e-58d8b4683032,False,2025-07-22T16:57:00.000Z,False,e9114dcb-819d-406a-b177-b0d7dfe805fd,e9114dcb-819d-40

---
## 🔍 Exploration Summary (for model design)

### What we learned
- Content volume is significant and structurally richer than people/site/file.
- `content_type` and `content_sub_type` appear strongly correlated and conflict-free in current dump.
- `content_site_id` / `createdbyid` / `lastmodifiedbyid` references are clean against raw site/people IDs.
- `content_files` is a nested list-of-dicts (`file_id`, etc.), not plain IDs.
- Most file references from `content_files` point to **image files** (excluded in your file pipeline), so direct file-link coverage to non-image files is low.
- `file_content_id` on non-image files only partially aligns with `content.id`, so future reconciliation logic should combine both signals.
- Page category name collisions exist globally; modeling as **site-scoped `SitePageCategory`** is justified.
- `_aggregated_locations` should be retained as-is for now.
- `content_topics` is deferred by design and should be ignored in initial extraction/ingestion.

### Proposed next-step decisions to confirm before we build extractor/loader
1. `:Content` node exact property list (include/exclude `content_type`, `content_sub_type`, `content_validated_on`).
2. Whether to ingest `LAST_MODIFIED` relation from `lastmodifiedbyid` in addition to `CREATED`.
3. `SitePageCategory` identity choice:
   - `(site_id, page_category_id)`
   - or `(site_id, page_category_name)`
   - or both
4. How to handle rows missing page category (`null/null`):
   - no category edge, OR
   - route to `_UNASSIGNED_` per site.
5. File linkage strategy for phase-1 content ingestion:
   - Use `content_files[*].file_id` only (to non-image files that exist), OR
   - also add a soft link using `file_content_id` matches, OR
   - defer all content↔file edges to a separate reconciliation phase.
